# Week 6 — Financial Mathematics and Option Pricing

> Part of the open-source teaching project **quant-math-roadmap**.
> For **education and research methodology only** — not investment advice; no result here represents a profitable or investable strategy.

## Learning objectives

- Compute discount factors, present values, and bond prices.
- Plot payoff diagrams for calls, puts, and simple combinations.
- Price European options with a binomial tree.
- Run sensitivity analyses over strike, volatility, maturity, and interest rate.

## Estimated study time

About 8–10 hours.

## Prerequisites

- Basic algebra and exponents
- Return concepts from Week 1

## External resources

- [NTU OpenCourseWare: Fundamentals of Financial Literacy](https://ocw.aca.ntu.edu.tw/courses/110S204)

> External resources are linked for reference only; this project does not reproduce any copyrighted course material.

In [ ]:
# Teaching style setup (deterministic look, consistent figures)
import matplotlib as _mpl
_mpl.rcParams['axes.unicode_minus'] = False
_mpl.rcParams['figure.figsize'] = (8.5, 4.5)
_mpl.rcParams['savefig.dpi'] = 100
import numpy as _np
_np.random.seed(0)  # belt-and-braces; library functions take explicit seeds

## Concepts

### The time value of money

A future cash flow must be **discounted** before it can be compared with money today. With annual rate $r$, $t$ years out, compounded $m$ times per year, the discount factor is $(1 + r/m)^{-mt}$. The present value is the sum of each cash flow times its discount factor.

### Option payoffs and binomial pricing

At expiry, a European call/put pays $\max(S-K,0)$, $\max(K-S,0)$.

This roadmap **deliberately uses only the binomial tree**: it needs nothing beyond arithmetic and the no-arbitrage idea — **no** stochastic calculus, and **no** Black-Scholes derivation required. The option value is the discounted expected value of its expiry payoff under the risk-neutral probability.

> Reminder: model prices should **not** be read as forecasts of real market prices.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quant_math_roadmap.finance.fixed_income import (
    bond_price, discount_factor, present_value, zero_coupon_bond_price,
)
from quant_math_roadmap.finance.derivatives import (
    binomial_european_option, call_payoff, put_payoff,
    long_straddle_payoff, put_call_parity_gap,
)

### A present-value calculator

In [ ]:
rate = 0.04
cash_flows = [100, 100, 100, 1100]  # 4-year bond, annual coupons
times = [1, 2, 3, 4]
pv = present_value(cash_flows, times, rate)
print(f'At a {rate:.0%} discount rate, the present value of the cash flows = {pv:.2f}')
for t in times:
    print(f'  t={t}: discount factor = {discount_factor(rate, t):.4f}')

### Bond pricing: price falls as the yield rises

In [ ]:
yields = np.linspace(0.01, 0.10, 50)
prices = [bond_price(1000, 0.05, 10, y, coupons_per_year=2) for y in yields]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(yields, prices, label='10-year, 5% coupon bond')
ax.axhline(1000, linestyle='--', label='Face value = 1000')
ax.set_title('Bond price vs yield')
ax.set_xlabel('Yield to maturity')
ax.set_ylabel('Bond price')
ax.legend()
plt.show()
zcb = zero_coupon_bond_price(1000, 10, 0.05)
print(f'Price of a 10-year zero-coupon bond (5% yield) = {zcb:.2f}')

As the yield rises, the bond price falls; when the coupon rate equals the yield, the bond prices at par (face value).

### Duration and convexity: a bond's interest-rate sensitivity

Knowing how the price moves with the yield is more useful than knowing a single price:

- **Macaulay duration**: the present-value-weighted average arrival time of the cash flows (in years). For a zero-coupon bond the duration equals the maturity exactly.
- **Modified duration** $D_{mod} = -\frac{1}{P}\frac{dP}{dy}$: for each percentage-point move in the yield, the price moves by roughly $D_{mod}$%.
- **Convexity** $C = \frac{1}{P}\frac{d^2P}{dy^2}$: the curvature correction; the second-order approximation is $\Delta P/P \approx -D_{mod}\Delta y + \tfrac12 C (\Delta y)^2$.

In [ ]:
from quant_math_roadmap.finance.fixed_income import (
    bond_convexity, macaulay_duration, modified_duration,
)

args = dict(face_value=1000, coupon_rate=0.05,
            years_to_maturity=10, yield_to_maturity=0.04)
mac = macaulay_duration(**args)
mod = modified_duration(**args)
conv = bond_convexity(**args)
print(f'Macaulay duration = {mac:.4f} years')
print(f'Modified duration = {mod:.4f}')
print(f'Convexity         = {conv:.4f}')

# Check what these numbers mean: true repricing vs first/second-order approximations
p0 = bond_price(1000, 0.05, 10, 0.04, coupons_per_year=2)
dy = 0.01  # yield +100bp
p1 = bond_price(1000, 0.05, 10, 0.04 + dy, coupons_per_year=2)
actual = p1 / p0 - 1
first_order = -mod * dy
second_order = -mod * dy + 0.5 * conv * dy**2
print(f'Actual price change          = {actual:+.4%}')
print(f'First order (duration)       = {first_order:+.4%}')
print(f'Second order (+convexity)    = {second_order:+.4%}  <- closer to the actual change')

Zero-coupon check: `macaulay_duration(1000, 0.0, 5, 0.04, coupons_per_year=1)` returns exactly 5.0 years — the only cash flow arrives at maturity.

### Payoff diagrams

In [ ]:
spot_grid = np.linspace(50, 150, 200)
strike = 100.0
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].plot(spot_grid, call_payoff(spot_grid, strike))
axes[0].set_title('Call payoff (K=100)')
axes[1].plot(spot_grid, put_payoff(spot_grid, strike))
axes[1].set_title('Put payoff (K=100)')
axes[2].plot(spot_grid, long_straddle_payoff(spot_grid, strike))
axes[2].set_title('Long straddle payoff (K=100)')
for ax in axes:
    ax.set_xlabel('Underlying price at expiry S')
    ax.set_ylabel('payoff')
plt.tight_layout()
plt.show()

### Binomial European option pricing

In [ ]:
params = {'spot': 100.0, 'strike': 100.0, 'rate': 0.05,
          'volatility': 0.20, 'maturity': 1.0}
call = binomial_european_option(**params, n_steps=300, option_type='call')
put = binomial_european_option(**params, n_steps=300, option_type='put')
print(f'European call price = {call:.4f}')
print(f'European put  price = {put:.4f}')
gap = put_call_parity_gap(call, put, params['spot'], params['strike'],
                          params['rate'], params['maturity'])
print(f'put-call parity residual = {gap:.6f}  (should be close to 0)')

### Sensitivity analysis

In [ ]:
vols = np.linspace(0.05, 0.6, 40)
call_by_vol = [binomial_european_option(100, 100, 0.05, v, 1.0,
               n_steps=200) for v in vols]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(vols, call_by_vol, label='ATM call (S=K=100)')
ax.set_title('European call price vs volatility')
ax.set_xlabel('Volatility proxy sigma')
ax.set_ylabel('Call price')
ax.legend()
plt.show()
print('Higher volatility makes the option more expensive — greater uncertainty benefits the buyer.')

### American options: the difference one max makes

An American option can be **exercised early**. On a binomial tree, that just changes the backward-induction value at each node to `max(discounted expected continuation value, immediate exercise value)`. Two classic results can be verified numerically:

1. **An American call on a non-dividend-paying underlying equals the European call** (early exercise never pays);
2. **An American put ≥ the European put** (deep in the money, receiving the strike early has time value).

In [ ]:
from quant_math_roadmap.finance.derivatives import binomial_american_option

common = dict(spot=100.0, strike=110.0, rate=0.06,
              volatility=0.2, maturity=2.0, n_steps=300)
eu_call = binomial_european_option(option_type='call', **common)
am_call = binomial_american_option(option_type='call', **common)
eu_put = binomial_european_option(option_type='put', **common)
am_put = binomial_american_option(option_type='put', **common)
print(f'European call = {eu_call:.4f} | American call = {am_call:.4f}  (equal)')
print(f'European put  = {eu_put:.4f} | American put  = {am_put:.4f}  (American is worth more)')
print(f'Early-exercise premium of the American put = {am_put - eu_put:.4f}')

### Greeks: price sensitivity to each input

The **Greeks** answer the question: if one input moves a little, how much does the price move? `binomial_greeks()` estimates them directly on the tree with finite differences:

| Greek | Definition | Intuition |
|-------|------|------|
| delta | ∂V/∂S | how much the option gains when the underlying rises by 1 |
| gamma | ∂²V/∂S² | how fast delta itself changes |
| vega  | ∂V/∂σ | impact of a 1-unit rise in volatility |
| theta | −∂V/∂T | decay from one year of time passing |
| rho   | ∂V/∂r | impact of a 1-unit rise in the rate |

> Finite differences on a tree are an approximation of an approximation — the numbers jitter slightly. This is a teaching tool, not a production-grade pricer.

In [ ]:
from quant_math_roadmap.finance.derivatives import binomial_greeks

greeks_call = binomial_greeks(100, 100, 0.05, 0.2, 1.0, option_type='call')
greeks_put = binomial_greeks(100, 100, 0.05, 0.2, 1.0, option_type='put')
for name in ['delta', 'gamma', 'vega', 'theta', 'rho']:
    print(f'{name:>6}: call = {greeks_call[name]:>9.4f} | put = {greeks_put[name]:>9.4f}')
print()
print(f'delta_call - delta_put = {greeks_call['delta'] - greeks_put['delta']:.6f}'
      '  (put-call parity says this difference is exactly 1)')

## Exercises

Work through these in order. **Basic exercises** consolidate the definitions, **applied exercises** are hands-on coding, and the **reflection question** connects the mathematics to backtesting and research methodology.

> The main notebook ships runnable starter code for each coding exercise. Full reference answers live in the matching `_solution` notebook under `notebooks/en/solutions/`.

### Basic exercises

1. In one sentence, explain why future money must be discounted.
2. Why do bond prices and yields move in opposite directions?
3. Describe the shape of a long straddle payoff and what it is betting on.

### Applied exercises

In [ ]:
# Applied exercise 1: compute the binomial call price as a function of strike (other parameters fixed),
# and confirm that a higher strike makes the call cheaper.
strikes = np.linspace(80, 120, 20)
call_by_strike = None  # TODO: [binomial_european_option(100, k, 0.05, 0.2, 1.0, n_steps=150) for k in strikes]
if call_by_strike is not None:
    print('Decreasing?', all(np.diff(call_by_strike) < 0))

In [ ]:
# Applied exercise 2: verify that the binomial call price converges (settles down) as the step count grows.
for steps in [10, 50, 200, 800]:
    price = None  # TODO: binomial_european_option(100, 100, 0.05, 0.2, 1.0, n_steps=steps)
    print(steps, price)

### Reflection question

1. Binomial model prices almost never match real market prices exactly. What caution does this suggest for designing trading strategies around model prices?

## Quiz (self-check)
Answer the multiple-choice questions, then run the next cell to check yourself. Answers are stored as hashes, not plaintext.

**Q1. When the yield rises, the bond price?**
- A. Rises
- B. Falls
- C. Stays the same
- D. Depends on the coupon

**Q2. The Macaulay duration of a zero-coupon bond equals?**
- A. 0
- B. The years to maturity
- C. The yield to maturity
- D. The coupon rate

**Q3. For a non-dividend-paying underlying, the American call price relative to the European call is?**
- A. Higher
- B. Equal
- C. Lower
- D. It depends

**Q4. What does positive convexity mean?**
- A. When yields fall, the price gain exceeds the linear duration estimate
- B. Price is proportional to yield
- C. The bond has default risk
- D. Duration is negative

In [ ]:
my_answers = {1: None, 2: None, 3: None, 4: None}  # TODO: fill in 'A' / 'B' / 'C' / 'D'

import hashlib as _hashlib
_expected = {1: 'e572273e26af1c17', 2: '9ec4573f1c035f2f', 3: '32d10537ba48c5ce', 4: '28fd59f43f42474e'}
_n_correct = 0
for _q, _ans in my_answers.items():
    if _ans is None:
        print(f'Q{_q}: unanswered')
        continue
    _h = _hashlib.sha256(f'qmr-w6-q{_q}-{str(_ans).strip().upper()}'.encode()).hexdigest()[:16]
    _ok = _h == _expected[_q]
    _n_correct += int(_ok)
    print(f'Q{_q}: ' + ('✔ correct' if _ok else '✘ incorrect'))
print(f'Score: {_n_correct} / {len(my_answers)}')

## Common mistakes

- **Mixing up compounding frequencies when discounting (annual vs semiannual).**
- **Confusing the option payoff (realized only at expiry) with the option price today.**
- **Treating the binomial price as exact when the step count is too small.**
- **Claiming the model price equals the real market price.**

## After this week, you should be able to

- [ ] Compute present values and bond prices.
- [ ] Plot and interpret call/put/straddle payoff diagrams.
- [ ] Price a European option with a binomial tree and explain each step.
- [ ] Run sensitivity analyses over the main parameters.

## References and attribution

- Every explanation, example and exercise in this notebook is **original** to this project.
- Recommended external resources: [`docs/resources.md`](../../docs/resources.md).
- Concept notes: [`docs/math/`](../../docs/math/) and [`docs/finance/`](../../docs/finance/).

### Privacy and disclaimer

- This notebook contains no real personal information.
- This notebook uses only reproducible synthetic data and needs no network access.
- This notebook makes no claim of real-world trading profitability.